## Data Ingestion and Preprocessing

In [17]:
# Import modules

from langchain_community.document_loaders import TextLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from google import genai
import os

In [2]:
load_dotenv()

True

In [3]:
# LLM 

llm = ChatGroq(model="openai/gpt-oss-120b")
llm.invoke("What is the capital of France?").content

'The capital of France is **Paris**.'

## Loading Documets

In [4]:
# Load PDF file
os.getcwd()
file_path = os.path.join(os.getcwd(), "data", "sample.pdf")
loader = PyPDFLoader(file_path)
documents = loader.load()

In [5]:
# Text Splitter
text_spliter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 150,
    length_function = len
)

In [7]:
#Split the documents into chunks
docs = text_spliter.split_documents(documents)
docs[0].metadata
docs[0].page_content

'Llama 2: Open Foundation and Fine-Tuned Chat Models\nHugo Touvron∗ Louis Martin† Kevin Stone†\nPeter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra\nPrajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen\nGuillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller\nCynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou\nHakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev'

In [9]:
# Create embeddings and vectorstore

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
    # With the `text-embedding-3` class
    # of models, you can specify the size
    # of the embeddings you want returned.
    # dimensions=1024
)
vectorstore = FAISS.from_documents(docs, embeddings)


In [27]:
# Similarity Search
vectorstore.similarity_search("what is a llama2 llm model?",k =2)

[Document(id='3ed4b05c-3859-4a8f-b83d-a23c1c08a233', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'c:\\Users\\saika\\Downloads\\AI\\Github\\SimpleRagChatBoatLangchain\\notebook\\data\\sample.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1'}, page_content='Our fine-tuned LLMs, calledLlama 2-Chat, are optimized for dialogue use cases. Our\nmodels outperform open-source chat models on most benchmarks we tested, and based on\nour human evaluations for helpfulness and safety, may be a suitable substitute for closed-\nsource models. We provide a detailed description of our approach to fine-tuning and safety\nimprovements ofLlama 2-Chatin order to enable the community to build on o

In [25]:
#Retreval
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":2})
print(retriever)

tags=['FAISS', 'OpenAIEmbeddings'] vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001E05F113770> search_kwargs={'k': 2}


In [13]:
# Promopt Template
prompt_template = """
        Answer the question based on the context provided below. 
        If the context does not contain sufficient information, respond with: 
        "I do not have enough information about this."

        Context: {context}

        Question: {question}

        Answer:"""

In [15]:
prompt=PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

In [22]:
parser=StrOutputParser()

def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

In [23]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [21]:
print(retriever)

tags=['FAISS', 'OpenAIEmbeddings'] vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001E05F113770> search_kwargs={'k': 2}


In [20]:
rag_chain.invoke("tell  me about the llama2 finetuning benchmark experiments?")

'The\u202fLlama\u202f2‑Chat models were evaluated through a series of benchmark experiments that focused on three main stages of finetuning:\n\n| Model size | Supervised‑finetuning (SFT) scores* | Reward‑modeling (RM) scores* | RLHF scores* |\n|------------|--------------------------------------|------------------------------|--------------|\n| 7\u202fB  | 0.28\u202f–\u202f0.51 (across 18 tasks) | — | — |\n| 13\u202fB | 0.24\u202f–\u202f0.66 (across 18 tasks) | — | — |\n| 34\u202fB | 0.27\u202f–\u202f0.57 (across 18 tasks) | — | — |\n| 70\u202fB | 0.31\u202f–\u202f0.65 (across 18 tasks) | — | — |\n\n\\*The numbers are the per‑task performance metrics reported in the paper (higher is better). They reflect the models’ results after each finetuning step.\n\n### Experiment workflow\n1. **Supervised Fine‑Tuning (SFT)** – The base Llama\u202f2 models were first instruction‑tuned on a large, curated dataset of prompts and high‑quality responses. This stage establishes the core “chat” ability.